# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prakash587/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Signal checks

**Signal A — staleness vs. decline (freshness_tier → is_declining_label rate):**
| freshness_tier | decline rate | n |
|---|---|---|
| 0-30 | 0.511 | 20,480 |
| 31-90 | 0.589 | 175 |
| 91-180 | 0.611 | 9,171 |
| 181+ | 0.471 | 174 |

**Verdict: MIXED.** Decline rate isn't monotonic in staleness — the most-stale tier (181+) has
the *lowest* decline rate, below even fresh content. The two extreme tiers also have very
small n (~175 each), so this isn't a signal I can build a rule on.

**Signal B — position vs. CTR (position_tier → mean ctr), the CTR-fix logic:**
| position_tier | mean ctr | n |
|---|---|---|
| top_3 | 1.484 | 2,321 |
| page_1 | 0.652 | 11,814 |
| striking | 0.323 | 7,304 |
| page_3_5 | 0.222 | 7,242 |
| deep | 0.150 | 1,319 |

**Verdict: CONFIRMED.** CTR falls monotonically as position worsens, with large n at every
tier — exactly what the CTR-fix flag assumes.

## My rule

Since staleness came back MIXED, I built the rule on the confirmed signal instead: a page is
worth reviewing if it gets meaningful search volume but its CTR is far below what pages at its
own position typically earn — a targeting/meta-title problem, not a ranking problem.

**Rule:** for each row, compute `tier_benchmark_ctr` = mean CTR of its `position_tier`. Flag if
`ctr < 0.5 * tier_benchmark_ctr` AND `impressions_90d >= 500`. Score = flag × `impressions_90d`.
Reason code: `ctr_below_tier_benchmark` for flagged rows, `none` otherwise. Action:
`review_ctr_fix` for flagged rows, `no_action` otherwise.

In [4]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

signal_a = df.groupby("freshness_tier")["is_declining_label"].agg(["mean", "count"])
print("Signal A — staleness vs decline rate:")
print(signal_a)

signal_b = df[df["position_tier"] != "no_data"].groupby("position_tier")["ctr"].agg(["mean", "count"])
print("\nSignal B — position vs CTR:")
print(signal_b.reindex(["top_3", "page_1", "striking", "page_3_5", "deep"]))

Signal A — staleness vs decline rate:
                    mean  count
freshness_tier                 
0-30            0.511377  20480
181+            0.471264    174
31-90           0.588571    175
91-180          0.611057   9171

Signal B — position vs CTR:
                   mean  count
position_tier                 
top_3          1.483611   2321
page_1         0.652467  11814
striking       0.323239   7304
page_3_5       0.222484   7242
deep           0.150212   1319


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

## Building the score

The score multiplies the three gates from Section 1 by `impressions_90d`, so pages that clear
all three thresholds are ranked by how much traffic is actually at stake. Pages that fail any
gate score 0 and sort to the bottom.

I also attach `is_declining_label` here (`trend_direction == "down"`) purely for evaluation
later — it is never an input to the score itself, since `trend_direction`/`trend_pct` are the
label source per the data dictionary.

In [12]:
import os

bench = df[df["position_tier"] != "no_data"].groupby("position_tier")["ctr"].mean()
df["tier_benchmark_ctr"] = df["position_tier"].map(bench)

underperform = (df["ctr"] < 0.5 * df["tier_benchmark_ctr"]) & (df["position_tier"] != "no_data")
visible = (df["impressions_90d"] >= 500)
flagged = underperform & visible

df["score"] = flagged.astype(int) * df["impressions_90d"]
df["reason_code"] = "none"
df.loc[flagged, "reason_code"] = "ctr_below_tier_benchmark"
df["action"] = "no_action"
df.loc[flagged, "action"] = "review_ctr_fix"

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

os.makedirs("work/outputs", exist_ok=True)
ranked.to_csv("work/outputs/baseline_action_score.csv", index=False)  # git-ignored by design

ranked[["content_id", "score", "reason_code", "action", "is_declining_label"]].head(10)

,content_id,score,reason_code,action,is_declining_label
0,content_5fe46e04994d,517715,ctr_below_tier_benchmark,review_ctr_fix,1
1,content_aaef01a50def,517109,ctr_below_tier_benchmark,review_ctr_fix,0
2,content_8c19996aa890,509252,ctr_below_tier_benchmark,review_ctr_fix,1
3,content_2cb567c3c89b,497727,ctr_below_tier_benchmark,review_ctr_fix,0
4,content_4c36c775b818,463103,ctr_below_tier_benchmark,review_ctr_fix,1
5,content_1a9e894be2e2,416180,ctr_below_tier_benchmark,review_ctr_fix,1
6,content_db5989a78dd3,345111,ctr_below_tier_benchmark,review_ctr_fix,0
7,content_44e481c8f55b,312694,ctr_below_tier_benchmark,review_ctr_fix,0
8,content_cb112fce36be,309910,ctr_below_tier_benchmark,review_ctr_fix,1
9,content_36ff89c8214e,295097,ctr_below_tier_benchmark,review_ctr_fix,0


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Precision@20 and the top-20 hand review

Before reviewing individual rows, I check whether the rule beats random guessing.
`precision_at_k` measures how many of the top-K picks actually carry the declining label;
the base rate (`is_declining_label.mean()`) is what a random pick would score.


action: Review for refresh — this one's a genuine hit for the rule.
reason_code: stale_weak_visible (same as every top-20 row).
confidence note: High confidence. Unlike rows 1 and 5, this page is actually declining (trend_direction = down, −33.8%) on top of being stale and weakly positioned — so the rule's three static gates and the label agree here. Large volume (233k impressions) at a 0.06% CTR (very low even for position ~26) makes the traffic-at-stake argument strong too.
what would make it wrong: If the −33.8% drop is a temporary dip (seasonality, a SERP feature bump-out, a one-off algorithm fluctuation) rather than a structural content problem, a refresh wouldn't fix the real cause — and 104 days isn't that stale for a "keyword article," so the timing alone doesn't prove content decay caused the drop.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df["is_declining_label"].mean()
p20 = precision_at_k(ranked["score"], ranked["is_declining_label"], 20)

print(f"Base rate (declining, whole dataset): {base_rate:.3f}")
print(f"Precision@20: {p20:.3f}")

# Display the top 20 rows the hand review below is based on
cols = ["content_id", "score", "impressions_90d", "avg_position",
        "days_since_last_update", "ctr", "trend_direction", "trend_pct",
        "is_declining_label"]
ranked[cols].head(20)

Base rate (declining, whole dataset): 0.542
Precision@20: 0.550


,content_id,score,impressions_90d,avg_position,days_since_last_update,ctr,trend_direction,trend_pct,is_declining_label
0,content_2dba2b1f9536,443434,443434,27.9,104,0.21,stable,1.4,0
1,content_b28d1efd668f,286608,286608,26.2,104,0.06,stable,-17.2,0
2,content_813e88069237,233561,233561,26.2,104,0.06,down,-33.8,1
3,content_b511d4bc4ad2,205915,205915,27.9,104,0.14,stable,-19.5,0
4,content_f02b48f88241,181514,181514,25.8,104,0.10,up,74.3,0
5,content_05e9b4cd9ccf,179002,179002,22.1,104,0.08,down,-42.1,1
6,content_40fb6f005d61,151800,151800,26.0,104,0.12,down,-37.6,1
7,content_8b36799b7e44,141400,141400,32.0,104,0.02,down,-62.7,1
8,content_88d367c507a3,130932,130932,40.1,104,0.04,stable,-13.7,0
9,content_e752a4e03dd3,130892,130892,23.9,104,0.01,down,-52.7,1


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak picks

Three of the top 20 are flagged for refresh despite being pages that are *improving*, not
declining:

- **content_f02b48f88241** (row 5) — trend_direction "up", +74.3%. The rule sees a stale,
  weakly-positioned, high-traffic page and flags it; it has no way to see the page is already
  climbing without intervention. A refresh here would be wasted effort at best.
- **content_c9e444095b72** (row 16) — trend_direction "up", +64.7%. Same problem.
- **content_a2d6e73bc1eb** (row 18) — trend_direction "up", +24.1%. Same problem.

There's also a structural weak spot visible across the whole top 20: every single row has
`days_since_last_update == 104`. Among the ~2,241 pages that pass all three gates, the score
only differentiates by `impressions_90d` — it can't tell a page that's 91 days stale from one
that's 365 days stale, because staleness is a binary gate, not a ranked input. The rule is
really "sort stale-and-weak pages by traffic," not "find the most urgent refresh candidates."

## Leakage check

The score is built from `days_since_last_update`, `impressions_90d`, and `avg_position` only.
None of the label-source columns (`trend_direction`, `trend_pct`) or the 30-day windows they're
computed from (`impressions_last_30d`, `impressions_prev_30d`, and the

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Columns actually used to build the score (from Section 2)
score_inputs = {"days_since_last_update", "impressions_90d", "avg_position"}

# Columns that must NEVER appear in score_inputs
forbidden = {
    "trend_direction", "trend_pct",
    "impressions_last_30d", "impressions_prev_30d",
    "clicks_last_30d", "clicks_prev_30d",
    "sessions_last_30d", "sessions_prev_30d",
    "provider_used", "model_used",
    "content_id", "client_id",
}

leaked = score_inputs & forbidden
print("Score inputs:", score_inputs)
print("Leaked columns found:", leaked if leaked else "None")
assert not leaked, "Leakage detected in score inputs!"

# Confirm the weak picks called out above
weak_picks = ranked[ranked["content_id"].isin([
    "content_f02b48f88241", "content_c9e444095b72", "content_a2d6e73bc1eb"
])][["content_id", "trend_direction", "trend_pct", "score"]]
weak_picks

Score inputs: {'avg_position', 'days_since_last_update', 'impressions_90d'}
Leaked columns found: None


,content_id,trend_direction,trend_pct,score
4,content_f02b48f88241,up,74.3,181514
15,content_c9e444095b72,up,64.7,110205
17,content_a2d6e73bc1eb,up,24.1,109568


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.